# 🎟️ Notebook 2: Sessions vs JWT

Once a user has logged in successfully, how do we *remember* that they are logged in across many requests? HTTP itself is stateless — every request is independent.

There are two popular answers:

1. 🟨 **Server-side sessions** — the server stores everything, and gives the client only a random ID.
2. 🟩 **JSON Web Tokens (JWT)** — the server gives the client a *signed* blob that it can verify on every request without any storage.

Neither is universally "best" — they trade off **revocation**, **size on the wire**, and **statelessness**.

## Learning objectives
- See both schemes work end-to-end, in plain Python.
- Understand the revocation problem with JWT.
- Know when to pick which.

## 🟨 Approach 1: Server-side session

The server keeps a dictionary `session_id -> user_info`. The client only stores the random session id (usually in a cookie). To log a user out, the server just deletes the entry.

In [ ]:
import secrets

# Pretend this is Redis or a database table.
SESSIONS = {}

def login_session(username):
    sid = secrets.token_urlsafe(16)
    SESSIONS[sid] = {"user": username}
    return sid  # send this to the client as a cookie

def whoami_session(sid):
    return SESSIONS.get(sid)

def logout_session(sid):
    SESSIONS.pop(sid, None)

sid = login_session("alice")
print("session id:", sid)
print("whoami:", whoami_session(sid))
logout_session(sid)
print("after logout:", whoami_session(sid))  # None — instantly revoked

## 🟩 Approach 2: JWT (stateless)

A JWT is just three base64url-encoded JSON parts joined with dots: **header.payload.signature**.

The signature is computed using a secret only the server knows. Anyone can *read* the payload (it is not encrypted!), but only the server can *create* a valid one.

The huge upside: **no server-side storage**. Any server with the secret can verify the token. The downside: you can't easily *revoke* a token before it expires.

In [ ]:
import jwt  # PyJWT
import time

SECRET = "super-secret-only-the-server-knows"

def login_jwt(username, ttl_seconds=60):
    payload = {
        "sub": username,
        "iat": int(time.time()),
        "exp": int(time.time()) + ttl_seconds,
    }
    return jwt.encode(payload, SECRET, algorithm="HS256")

def whoami_jwt(token):
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError as e:
        return f"invalid: {e}"

token = login_jwt("alice", ttl_seconds=2)
print("token:", token)
print("whoami:", whoami_jwt(token))

print("\nWaiting 3 seconds for the token to expire...")
time.sleep(3)
print("whoami:", whoami_jwt(token))  # invalid: Signature has expired

In [ ]:
# What if someone tampers with the payload? The signature won't match.
import base64, json

header, payload, sig = token.split(".")
decoded = json.loads(base64.urlsafe_b64decode(payload + "==").decode())
print("payload (anyone can read this):", decoded)

decoded["sub"] = "admin"
tampered_payload = base64.urlsafe_b64encode(json.dumps(decoded).encode()).decode().rstrip("=")
forged = f"{header}.{tampered_payload}.{sig}"
print("forged check:", whoami_jwt(forged))

## 🤔 Which to pick?

| Concern | Sessions | JWT |
|---|---|---|
| Revocation (force logout) | ✅ instant | ❌ must wait for expiry (or maintain a deny-list) |
| Server storage | ❌ needs Redis/DB | ✅ stateless |
| Size on the wire | ✅ small (just an id) | ❌ bigger (JSON payload) |
| Cross-service auth | meh — every service hits the session store | ✅ each service verifies with the public key |

**Rule of thumb:** classic web app with one backend → sessions are simpler. Microservices, mobile, or third-party APIs → JWT (often *short-lived* JWT + a long-lived refresh token).